In [5]:
%reload_ext autoreload
%autoreload 2

import warnings, os, sys
warnings.filterwarnings('ignore')
from IPython.display import clear_output
from pathlib import Path

import numpy as np
import pandas as pd
import pickle
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)
from tqdm.auto import tqdm
import json
import networkx as nx
from networkx.readwrite import json_graph

from rdkit import Chem
from rdkit.Chem import PandasTools
# PandasTools.RenderImagesInAllDataFrames(images=True)
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = 700, 500

sys.path.append(os.path.abspath(".."))
from new_geass import Geass
import utils
clear_output()

In [3]:
df_frags = pd.read_csv('/home/mattia/geass/NPA_fragmentation/NPA_fragments.csv')
frag_cols = ['Frag_IDs', 'Frag_ISOSMILES', 'Frag_Sim', 'Frag_Ann']
df_frags[frag_cols] = df_frags[frag_cols].apply(
    lambda s: s.str[1:-1].str.split(","))

# Explode
df_nodes = df_frags[~df_frags['Frag_ISOSMILES'].apply(lambda x: 'ERROR' in x)]
df_nodes = df_nodes.explode(frag_cols)
df_nodes['Frag_Sim'] = df_nodes['Frag_Sim'].astype(float)

# Build unique fragment set and RDKit molecules
df_nodes = df_nodes.drop_duplicates('Frag_ISOSMILES')
PandasTools.AddMoleculeColumnToFrame(df_nodes, 'Frag_ISOSMILES')

In [ ]:
from annotator import AnnotationEngine

sim_threshold = 0.01

def _annotate_with_method(mol, method):
    AnnotationEngine.method = method
    AnnotationEngine.initialize_resources()
    AnnotationEngine.annotate_fragment(mol, threshold=sim_threshold)

    return pd.Series({
        'ID': mol.GetProp('MUID') if mol.HasProp('MUID') else '',
        'Sim': float(mol.GetProp('Similarity')) if mol.HasProp('Similarity') else 0.0,
    })

map4_ann = df_nodes['ROMol'].parallel_apply(lambda m: _annotate_with_method(m, 'map4'))
df_nodes['Frag_ID_map4'] = map4_ann['ID']
df_nodes['Frag_Sim_map4'] = map4_ann['Sim']

atomp_ann = df_nodes['ROMol'].parallel_apply(lambda m: _annotate_with_method(m, 'atompair'))
df_nodes['Frag_ID_atomp'] = atomp_ann['ID']
df_nodes['Frag_Sim_atomp'] = atomp_ann['Sim']

# Map MUDB IDs to corresponding ISOSMILES
mudb = pd.read_pickle('/home/mattia/geass/data/MUDB.pkl')
id_to_isosmiles = mudb.drop_duplicates('ID').set_index('ID')['ISOSMILES']
df_nodes['Frag_ISOSMILES_map4'] = df_nodes['Frag_ID_map4'].map(id_to_isosmiles).fillna('')
df_nodes['Frag_ISOSMILES_atomp'] = df_nodes['Frag_ID_atomp'].map(id_to_isosmiles).fillna('')

df_nodes = df_nodes[['NPA_ID', 'ISOSMILES', 'Frag_ISOSMILES',
                     'Frag_ID_map4', 'Frag_Sim_map4', 'Frag_ISOSMILES_map4',
                     'Frag_ID_atomp', 'Frag_Sim_atomp', 'Frag_ISOSMILES_atomp']]
df_nodes.to_csv("/home/mattia/geass/NPA_fragmentation/NPA_nodes.csv")

In [ ]:
# Drop rows where both similarities are exactly 1
mask_both_one = (df_nodes['Frag_Sim_map4'] == 1.0) & (df_nodes['Frag_Sim_atomp'] == 1.0)
df_nodes_diff = df_nodes.loc[~mask_both_one].copy()

# Drop rows where map4 and atomp assigned the same fragment ID
df_nodes_diff = df_nodes_diff[
    df_nodes_diff['Frag_IDs_map4'] != df_nodes_diff['Frag_IDs_atomp']]

df_nodes_diff.to_csv("/home/mattia/geass/NPA_fragmentation/NPA_nodes_diff.csv")

After visual inspection, the best similarity thresholds are:
map4 >= 0.35
atompair >= 0.76

For map4 < 0.35, atompair assigns 66 fragments with sim >= 0.76
For atompair < 0.76, map4 assigns 1570 fragments with sim >= 0.35